### MAZE

In [15]:
import time
maze = [
    ["S", ".", "#", ".", "."],
    [".", ".", "#", ".", "#"],
    ["#", ".", ".", ".", "."],
    [".", "#", "#", ".", "."],
    [".", ".", ".", "#", "G"]
]

for row in maze:
    print(" ".join(row))


S . # . .
. . # . #
# . . . .
. # # . .
. . . # G


In [2]:

start = None
goal = None

for row in range(len(maze)):
    for col in range(len(maze[row])):

        if maze[row][col] == "S":
            start = (row, col)

        elif maze[row][col] == "G":
            goal = (row, col)

print("Start:", start)
print("Goal:", goal)


Start: (0, 0)
Goal: (4, 4)


In [3]:

directions = [
    (-1, 0),  # Up
    (1, 0),   # Down
    (0, -1),  # Left
    (0, 1)    # Right
]


In [4]:

def is_valid(maze, row, col):

    rows = len(maze)
    cols = len(maze[0])

    if row < 0 or row >= rows:
        return False

    if col < 0 or col >= cols:
        return False

    if maze[row][col] == "#":
        return False

    return True


In [18]:
def display_maze(maze, path):

    solved_maze = [row.copy() for row in maze]

    if path:
        for row, col in path:

            if solved_maze[row][col] not in ("S", "G"):
                solved_maze[row][col] = "*"

    for row in solved_maze:
        print(" ".join(row))

In [29]:

def reconstruct_path(parent, start, goal):

    if goal not in parent:
        return None

    path = []

    current = goal

    while current is not None:

        path.append(current)

        current = parent[current]

    path.reverse()

    return path


### Breadth-First Search (BFS)


In [16]:

from collections import deque


def bfs(maze, start, goal):

    queue = deque([start])
    visited = {start}
    parent = {start: None}

    explored = 0

    while queue:

        current = queue.popleft()
        explored += 1

        if current == goal:
            break

        row, col = current

        for dr, dc in directions:

            new_row = row + dr
            new_col = col + dc

            neighbor = (new_row, new_col)

            if is_valid(maze, new_row, new_col):

                if neighbor not in visited:

                    visited.add(neighbor)
                    parent[neighbor] = current
                    queue.append(neighbor)

    if goal not in parent:
        return None, explored

    return parent, explored


In [31]:

print("\nBFS Results:")

parent, explored = bfs(maze, start, goal)

if parent:

    path = reconstruct_path(parent, start, goal)

    display_maze(maze, path)

    print("Path:", path)

    print("Path Length:", len(path) - 1)

else:
    print("No path found!")



BFS Results:
S . # . .
* * # . #
# * * * .
. # # * *
. . . # G
Path: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (3, 3), (3, 4), (4, 4)]
Path Length: 8


### Depth-First Search (DFS)

In [20]:

def dfs(maze, start, goal):

    stack = [start]
    visited = {start}
    parent = {start: None}

    explored = 0

    while stack:

        current = stack.pop()
        explored += 1

        if current == goal:
            break

        row, col = current

        for dr, dc in directions:

            new_row = row + dr
            new_col = col + dc

            neighbor = (new_row, new_col)

            if is_valid(maze, new_row, new_col):

                if neighbor not in visited:

                    visited.add(neighbor)
                    parent[neighbor] = current
                    stack.append(neighbor)

    if goal not in parent:
        return None, explored

    return parent, explored


In [32]:

print("\nDFS Results:")

parent, explored = dfs(maze, start, goal)

if parent:

    path = reconstruct_path(parent, start, goal)

    display_maze(maze, path)

    print("Path:", path)

    print("Path Length:", len(path) - 1)

else:
    print("No path found!")



DFS Results:
S * # . .
. * # . #
# * * * *
. # # . *
. . . # G
Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
Path Length: 8


### A* Search

In [12]:

def heuristic(current, goal):

    return abs(current[0] - goal[0]) + abs(current[1] - goal[1])


In [34]:

import heapq


def astar(maze, start, goal):

    priority_queue = []
    heapq.heappush(priority_queue, (0, start))

    visited = set()
    parent = {start: None}
    g_cost = {start: 0}

    explored = 0

    while priority_queue:

        current_priority, current = heapq.heappop(priority_queue)

        if current in visited:
            continue

        visited.add(current)
        explored += 1

        if current == goal:
            break

        row, col = current

        for dr, dc in directions:

            new_row = row + dr
            new_col = col + dc

            neighbor = (new_row, new_col)

            if is_valid(maze, new_row, new_col):

                new_cost = g_cost[current] + 1

                if neighbor not in g_cost or new_cost < g_cost[neighbor]:

                    g_cost[neighbor] = new_cost
                    parent[neighbor] = current

                    f_cost = new_cost + heuristic(neighbor, goal)

                    heapq.heappush(
                        priority_queue,
                        (f_cost, neighbor)
                    )

    if goal not in parent:
        return None, explored

    return parent, explored


In [35]:

print("\nA* Results:")

parent, explored = astar(maze, start, goal)

if parent:

    path = reconstruct_path(parent, start, goal)

    display_maze(maze, path)

    print("Path:", path)

    print("Path Length:", len(path) - 1)

else:
    print("No path found!")


A* Results:
S * # . .
. * # . #
# * * * *
. # # . *
. . . # G
Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
Path Length: 8


### Comparisn:

Feature | BFS | DFS | A*
Search type | Uninformed | Uninformed | Informed
Data structure | Queue | Stack | Priority queue
Shortest path | Yes | Not guaranteed | Yes, with our heuristic
Heuristic | No | No | Yes
Search strategy | Level by level | Deep first | Lowest estimated cost


In [ ]:

algorithms = {
    "BFS": bfs,
    "DFS": dfs,
    "A*": astar
}

results = {}

repetitions = 1000

for name, algorithm in algorithms.items():

    start_time = time.perf_counter()

    for _ in range(repetitions):
        parent, explored = algorithm(maze, start, goal)

    end_time = time.perf_counter()

    execution_time = (end_time - start_time) / repetitions

    if parent:
        path = reconstruct_path(parent, start, goal)
        path_length = len(path) - 1
    else:
        path_length = None

    results[name] = {
        "execution_time": execution_time,
        "explored_nodes": explored,
        "path_length": path_length
    }

print("\n========== ALGORITHM COMPARISON ==========")

for name, result in results.items():

    print(f"\n{name}")

    print(
        f"Execution Time: "
        f"{result['execution_time']:.8f} seconds"
    )

    print(f"Explored Nodes: {result['explored_nodes']}")

    print(f"Path Length: {result['path_length']}")



========== ALGORITHM COMPARISON ==========

BFS
Execution Time: 0.00003281 seconds
Explored Nodes: 14
Path Length: 8

DFS
Execution Time: 0.00001776 seconds
Explored Nodes: 9
Path Length: 8

A*
Execution Time: 0.00002948 seconds
Explored Nodes: 11
Path Length: 8
